# <font color="#418FDE" size="6.5" uppercase>**Transfer mit PyTorch**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Setzen Bildaugmentation, Dropout, Batch Normalization, Scheduler und Early Stopping kontrolliert ein. 
- Nutzen ein leichtes vortrainiertes torchvision-Modell als eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Modelle hinsichtlich Genauigkeit, Laufzeit, Größe und Offline-Fallback. 


## **1. Regularisierung kontrolliert einsetzen**

### **1.1. Bildaugmentation gezielt nutzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_01.jpg?v=1787664402" width="250">



>* Augmentation macht CNNs robuster durch plausible Variationen
>* Wähle Veränderungen passend zur Anwendung

>* Augmentation passend zur Aufgabe wählen
>* Hilft bei kleinen, unausgewogenen Datensätzen

>* Nur Trainingsdaten augmentieren, Validierung unverändert lassen
>* Stärke visuell prüfen, realistische Varianten wählen



In [ ]:
#@title Python-Code - Bildaugmentation gezielt nutzen

# Dieses Beispiel zeigt gezielte Bildaugmentation.
# Plausible Änderungen erhalten die Bildklasse.
# Die Grafik vergleicht kontrollierte Varianten.

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from PIL import ImageEnhance

# Wir erzeugen ein kleines synthetisches RGB-Bild.
image_size = 96
base_image = np.full((image_size, image_size, 3), 235, dtype=np.uint8)

# Ein einfaches Blattmotiv dient als erkennbare Klasse.
y_grid, x_grid = np.ogrid[:image_size, :image_size]
leaf_mask = ((x_grid - 48) / 28) ** 2 + ((y_grid - 48) / 18) ** 2 <= 1
base_image[leaf_mask] = np.array([45, 150, 70], dtype=np.uint8)

# Eine Mittelrippe macht die Orientierung sichtbar.
base_image[46:50, 24:73] = np.array([25, 95, 45], dtype=np.uint8)

# Pillow wendet typische Trainingsaugmentationen im Speicher an.
pil_image = Image.fromarray(base_image)
rotated = pil_image.rotate(12, resample=Image.Resampling.BILINEAR, fillcolor=(235, 235, 235))

# Helligkeitsschwankungen simulieren unterschiedliche Beleuchtung.
brighter = ImageEnhance.Brightness(pil_image).enhance(1.35)

# Ein leichter Ausschnitt simuliert ungenaue Zentrierung.
cropped = pil_image.crop((8, 8, 88, 88)).resize((96, 96), Image.Resampling.BILINEAR)

# Wir kombinieren die Varianten in einem einzigen Vergleichsbild.
variants = [pil_image, rotated, brighter, cropped]
labels = ["Original", "Rotation 12°", "Heller", "Ausschnitt"]
canvas = np.full((126, 384, 3), 255, dtype=np.uint8)

# Jede Variante bleibt für Menschen als Blatt erkennbar.
for index, variant in enumerate(variants):
    left = index * 96
    canvas[:96, left:left + 96] = np.array(variant)

print("Bildaugmentation gehört nur in den Trainingsdatensatz.")
print("Alle gezeigten Varianten bleiben plausibel dieselbe Klasse: Blatt.")
print("Zu starke oder fachlich falsche Änderungen würden das Label verfälschen.")

# Eine einzige Achse zeigt alle Varianten nebeneinander.
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(canvas)
ax.set_title("Kontrollierte Bildaugmentation an einem synthetischen Blatt")

# Textlabels erklären die jeweilige Transformation.
for index, label in enumerate(labels):
    ax.text(index * 96 + 48, 113, label, ha="center", va="center", fontsize=9)

ax.set_xlabel("Augmentationsvariante")
ax.set_ylabel("Pixelposition")
ax.set_xticks([])
ax.set_yticks([])
plt.show()



### **1.2. Dropout und BatchNorm**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_02.jpg?v=1787664400" width="250">



>* Dropout verhindert Abhängigkeit von einzelnen Aktivierungen
>* Gezielt einsetzen und Validierung beobachten

>* BatchNorm stabilisiert Aktivierungen und beschleunigt Training
>* Trainings- und Evaluationsmodus korrekt setzen

>* BatchNorm in Faltungsblöcken, Dropout im Kopf
>* Transfer-Statistiken schützen, Wirkung systematisch prüfen



In [ ]:
#@title Python-Code - Dropout und BatchNorm

# Dieses Beispiel zeigt Dropout und BatchNorm praktisch.
# Wir vergleichen Trainingsmodus und Auswertungsmodus.
# Die Ausgabe macht zufällige Aktivierungen sichtbar.

import torch
import matplotlib.pyplot as plt

# Feste Zufallszahlen machen das Beispiel reproduzierbar.
torch.manual_seed(42)

# Vier kleine Aktivierungsvektoren stehen für Mini-Batch-Merkmale.
features = torch.tensor(
    [[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [3.0, 6.0, 9.0], [4.0, 8.0, 12.0]]
)

# BatchNorm normalisiert jede Merkmalsspalte im Mini-Batch.
batch_norm = torch.nn.BatchNorm1d(num_features=3)

# Dropout deaktiviert im Training zufällig einzelne Aktivierungen.
dropout = torch.nn.Dropout(p=0.5)

# Im Trainingsmodus sind BatchNorm und Dropout aktiv.
batch_norm.train()
dropout.train()

normalized_train = batch_norm(features)
dropped_train = dropout(normalized_train)

# Im Auswertungsmodus nutzt BatchNorm gespeicherte Werte.
batch_norm.eval()
dropout.eval()

normalized_eval = batch_norm(features)
dropped_eval = dropout(normalized_eval)

# Diese Prüfung schützt vor unerwarteten Formfehlern.
if dropped_train.shape != features.shape:
    raise ValueError("Die Ausgabeform passt nicht zu den Eingaben.")

train_zero_share = (dropped_train == 0).float().mean().item()
eval_zero_share = (dropped_eval == 0).float().mean().item()

print("PyTorch-Version:", torch.__version__)
print("Dropout-Nullanteil im Training:", round(train_zero_share, 2))
print("Dropout-Nullanteil in der Auswertung:", round(eval_zero_share, 2))
print("BatchNorm-Mittelwert im Training:", round(normalized_train.mean().item(), 2))
print("BatchNorm-Mittelwert in der Auswertung:", round(normalized_eval.mean().item(), 2))

# Das Diagramm vergleicht dieselbe Aktivierung in beiden Modi.
fig, ax = plt.subplots(figsize=(7, 4))

positions = [0, 1, 2]
ax.bar(positions, dropped_train[0].detach().numpy(), label="Training")

ax.plot(positions, dropped_eval[0].detach().numpy(), marker="o", label="Auswertung")
ax.set_title("Dropout und BatchNorm: Modus macht den Unterschied")

ax.set_xlabel("Merkmal")
ax.set_ylabel("Aktivierungswert")

ax.set_xticks(positions)
ax.legend()

plt.show()



### **1.3. Lernrate und Stopp**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_01_03.jpg?v=1787664404" width="250">



>* Lernrate steuert Schrittgröße und Trainingsstabilität
>* Scheduler passt Tempo für grobes und feines Lernen an

>* Validierungsleistung steuert sinnvolle Lernraten-Senkung
>* Lernrate mit Regularisierung gemeinsam betrachten

>* Validierungsmetriken zeigen den richtigen Stoppzeitpunkt
>* Geduld verhindert Überanpassung und unnötiges Training



In [ ]:
#@title Python-Code - Lernrate und Stopp

# Dieses Beispiel zeigt Lernrate und Early Stopping.
# Ein Scheduler senkt die Lernrate automatisch.
# Der beste Validierungsverlust beendet das Training.

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

# Wir erzeugen kleine synthetische Bilddaten.
torch.manual_seed(42)
rng = np.random.default_rng(42)

# Jede Klasse hat ein helles Quadrat an anderer Stelle.
sample_count = 240
image_size = 8

images = rng.normal(0.0, 0.15, size=(sample_count, 1, image_size, image_size))
labels = np.zeros(sample_count, dtype=np.int64)

for index in range(sample_count):
    label = index % 2
    labels[index] = label
    row_start = 1 if label == 0 else 4
    col_start = 1 if label == 0 else 4
    images[index, 0, row_start:row_start + 3, col_start:col_start + 3] += 1.0

# Wir mischen deterministisch und teilen in Training und Validierung.
order = rng.permutation(sample_count)
images = images[order].astype(np.float32)
labels = labels[order]

train_size = 180
x_train = torch.tensor(images[:train_size])
y_train = torch.tensor(labels[:train_size])

x_val = torch.tensor(images[train_size:])
y_val = torch.tensor(labels[train_size:])

if x_train.shape[0] != y_train.shape[0]:
    raise ValueError("Trainingsdaten und Labels passen nicht zusammen.")

# Ein kleines CNN reicht für das Lernraten-Beispiel.
model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.BatchNorm2d(4),
    nn.ReLU(),
    nn.Flatten(),
    nn.Dropout(p=0.25),
    nn.Linear(4 * image_size * image_size, 2),
)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.08)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

# Early Stopping beobachtet den Validierungsverlust.
best_val_loss = float("inf")
best_state = None
patience = 5
wait = 0

history_epochs = []
history_val_loss = []
history_lr = []

for epoch in range(1, 41):
    model.train()
    optimizer.zero_grad()
    train_logits = model(x_train)
    train_loss = loss_function(train_logits, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(x_val)
        val_loss = loss_function(val_logits, y_val).item()
        val_accuracy = (val_logits.argmax(1) == y_val).float().mean().item()

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    history_epochs.append(epoch)
    history_val_loss.append(val_loss)
    history_lr.append(current_lr)

    if val_loss < best_val_loss - 0.001:
        best_val_loss = val_loss
        best_state = model.state_dict()
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        break

if best_state is not None:
    model.load_state_dict(best_state)

print(f"Epochen bis Stopp: {history_epochs[-1]}")
print(f"Beste Validierungs-Loss: {best_val_loss:.3f}")
print(f"Letzte Validierungsgenauigkeit: {val_accuracy:.2f}")
print(f"Start-Lernrate: {history_lr[0]:.3f}")
print(f"End-Lernrate: {history_lr[-1]:.3f}")

# Die Kurve zeigt Verlust und Lernratenwechsel gemeinsam.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_epochs, history_val_loss, marker="o", label="Validierungsverlust")
ax.set_title("Scheduler und Early Stopping im Trainingsverlauf")
ax.set_xlabel("Epoche")
ax.set_ylabel("Validierungsverlust")

for epoch, lr in zip(history_epochs, history_lr):
    if epoch == 1 or lr != history_lr[epoch - 2]:
        ax.axvline(epoch, color="gray", alpha=0.25)

ax.legend()
plt.show()



## **2. Transfer Learning**

### **2.1. MobileNetV Small**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_01.jpg?v=1787664388" width="250">



>* Vortrainierte Merkmale statt Training von Grund auf
>* Leichtes Modell für kleine Datensätze

>* Effizient mit wenigen Parametern und Rechenaufwand
>* Geeignet für Laptops, Mobilgeräte und Edge-Einsatz

>* Nur der neue Klassifikationskopf wird trainiert
>* Gut für Fotos, begrenzt bei Spezialbildern



In [ ]:
#@title Python-Code - MobileNetV Small

# Dieses Beispiel zeigt MobileNetV3 als Merkmalsextraktor.
# Eingefrorene Schichten liefern feste Bildmerkmale.
# Nur der kleine Klassifikationskopf bleibt trainierbar.

import numpy as np
import torch
import matplotlib.pyplot as plt

# Wir verwenden CPU und feste Zufallszahlen.
torch.manual_seed(42)
rng = np.random.default_rng(42)

# Kleine synthetische RGB-Bilder ersetzen echte Kursdaten.
image_count = 8
image_size = 64

# Die Bilder bleiben klein und vollständig im Speicher.
images = rng.integers(
    low=0,
    high=256,
    size=(image_count, 3, image_size, image_size),
    dtype=np.uint8,
)

# Eine einfache Prüfung macht die Annahmen sichtbar.
if images.shape != (8, 3, 64, 64):
    raise ValueError("Die Bildform passt nicht zum Beispiel.")

# Pixelwerte werden wie bei CNNs üblich skaliert.
image_tensor = torch.tensor(images, dtype=torch.float32) / 255.0

# Dieses kleine Netz imitiert einen MobileNet-Merkmalsblock.
feature_extractor = torch.nn.Sequential(
    torch.nn.Conv2d(3, 8, kernel_size=3, padding=1),
    torch.nn.BatchNorm2d(8),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
)

# Alle Merkmalsextraktor-Parameter werden eingefroren.
for parameter in feature_extractor.parameters():
    parameter.requires_grad = False

# Nur dieser Kopf würde für neue Klassen trainiert.
classifier_head = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Dropout(p=0.2),
    torch.nn.Linear(8, 3),
)

# Wir zählen trainierbare und eingefrorene Parameter.
frozen_count = sum(p.numel() for p in feature_extractor.parameters())
trainable_count = sum(p.numel() for p in classifier_head.parameters())

# Ein Vorwärtslauf zeigt die Form der festen Merkmale.
with torch.no_grad():
    features = feature_extractor(image_tensor)
    logits = classifier_head(features)

# Die Ausgabe bleibt kurz und unterstützt die Kernaussage.
print("Beispiel: MobileNetV3-Small-Prinzip ohne Gewichtsdownload.")
print(f"Eingefrorene Parameter im Merkmalsextraktor: {frozen_count}")
print(f"Trainierbare Parameter im neuen Kopf: {trainable_count}")
print(f"Merkmalsform pro Bild: {tuple(features.shape)}")
print(f"Logit-Form für 3 Zielklassen: {tuple(logits.shape)}")

# Das Diagramm vergleicht eingefrorene und trainierbare Teile.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["eingefroren", "trainierbar"], [frozen_count, trainable_count])
ax.set_title("Transfer Learning: nur der Kopf lernt")
ax.set_ylabel("Parameteranzahl")
plt.show()



### **2.2. Schichten einfrieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_02.jpg?v=1787664390" width="250">



>* Vortrainierte Schichten bleiben unverändert
>* Nur der neue Klassifikationskopf lernt

>* Schnelleres Training mit weniger Speicherbedarf
>* Weniger Overfitting, klarere Modellrollen

>* Einfrieren je nach Domänenähnlichkeit wählen
>* Spätere Schichten bei Bedarf feinjustieren



In [ ]:
#@title Python-Code - Schichten einfrieren

# Dieses Beispiel zeigt eingefrorene Schichten in PyTorch.
# Der Merkmalsextraktor bleibt unverändert beim Training.
# Nur der neue Klassifikationskopf erhält Gradienten.

import torch
import matplotlib.pyplot as plt

# Wir verwenden CPU und feste Zufallszahlen.
torch.manual_seed(42)

# Ein kleines CNN ersetzt hier ein großes vortrainiertes Modell.
feature_extractor = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
)

# Diese Schichten werden wie vortrainierte Merkmale eingefroren.
for parameter in feature_extractor.parameters():
    parameter.requires_grad = False

# Der neue Kopf lernt die Klassen der eigenen Aufgabe.
classifier_head = torch.nn.Linear(4, 2)
model = torch.nn.Sequential(feature_extractor, torch.nn.Flatten(), classifier_head)

# Synthetische Mini-Bilder halten das Beispiel vollständig offline.
images = torch.randn(8, 3, 16, 16)
labels = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1])

# Wir speichern Gewichte vor einem Trainingsschritt.
frozen_before = feature_extractor[0].weight.detach().clone()
head_before = classifier_head.weight.detach().clone()

# Nur trainierbare Parameter werden an den Optimierer übergeben.
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(trainable_parameters, lr=0.2)

# Ein einzelner Schritt reicht, um den Unterschied zu sehen.
logits = model(images)
loss = torch.nn.functional.cross_entropy(logits, labels)
loss.backward()
optimizer.step()

# Wir messen, welche Gewichte sich verändert haben.
frozen_change = (feature_extractor[0].weight - frozen_before).abs().max().item()
head_change = (classifier_head.weight - head_before).abs().max().item()

# Die Ausgabe zeigt die Wirkung des Einfrierens.
print(f"Eingefrorene Conv-Gewichte verändert: {frozen_change:.6f}")
print(f"Kopf-Gewichte verändert: {head_change:.6f}")
print(f"Trainierbare Parametergruppen: {len(trainable_parameters)}")

# Das Balkendiagramm macht den Vergleich sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["eingefrorene Schicht", "neuer Kopf"], [frozen_change, head_change])
ax.set_title("Gewichtsänderung nach einem Trainingsschritt")
ax.set_ylabel("maximale absolute Änderung")

# Nur der Kopf sollte eine sichtbare Änderung zeigen.
ax.set_xlabel("Modellteil")
plt.show()



### **2.3. Klassifikationskopf trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_02_03.jpg?v=1787664392" width="250">



>* Eingefrorenes MobileNet erkennt allgemeine Bildmuster
>* Neuer Kopf ordnet Merkmale Zielklassen zu

>* Spart Rechenzeit, Speicher und Trainingsdaten
>* Senkt Überanpassung bei kleinen Datensätzen

>* Kopf und Daten passend vorbereiten
>* Validierung prüft Generalisierung und Überanpassung



In [ ]:
#@title Python-Code - Klassifikationskopf trainieren

# Wir trainieren nur einen kleinen Klassifikationskopf.
# Der eingefrorene Teil liefert feste Bildmerkmale.
# Die Ausgabe zeigt lernende Kopfparameter.

import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Feste Zufallszahlen machen das Beispiel reproduzierbar.
np.random.seed(42)
torch.manual_seed(42)

# Synthetische Bilder enthalten einfache helle Muster.
image_count = 240
image_size = 16
images = np.zeros((image_count, 1, image_size, image_size), dtype=np.float32)
labels = np.zeros(image_count, dtype=np.int64)

# Klasse null hat links, Klasse eins rechts helle Pixel.
for index in range(image_count):
    label = index % 2
    labels[index] = label
    noise = np.random.normal(0.0, 0.12, (image_size, image_size))
    images[index, 0] = noise

    if label == 0:
        images[index, 0, 4:12, 3:6] += 1.0
    else:
        images[index, 0, 4:12, 10:13] += 1.0

# Die Pixelwerte bleiben in einem sinnvollen Bereich.
images = np.clip(images, 0.0, 1.0)

# Training und Test werden sauber getrennt.
train_images, test_images, train_labels, test_labels = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Dieser kleine Backbone steht für einen Merkmalsextraktor.
backbone = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
)

# Wir setzen feste Filter statt sie zu trainieren.
with torch.no_grad():
    backbone[0].weight.zero_()
    backbone[0].bias.zero_()
    backbone[0].weight[0, 0, :, 0] = 1.0
    backbone[0].weight[1, 0, :, 2] = 1.0
    backbone[0].weight[2, 0, 0, :] = 1.0
    backbone[0].weight[3, 0, 2, :] = 1.0

# Alle Backbone-Parameter werden eingefroren.
for parameter in backbone.parameters():
    parameter.requires_grad = False

# Nur dieser Kopf lernt die neue Entscheidung.
head = nn.Sequential(
    nn.Dropout(p=0.1),
    nn.Linear(4, 2),
)

# Das Gesamtmodell kombiniert feste Merkmale und lernenden Kopf.
model = nn.Sequential(backbone, head)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(head.parameters(), lr=0.08)

# Tensoren werden einmal vorbereitet.
train_x = torch.tensor(train_images)
train_y = torch.tensor(train_labels)
test_x = torch.tensor(test_images)

# Wir prüfen die erwartete Merkmalsform.
feature_shape = backbone(train_x[:2]).shape
if feature_shape[1] != 4:
    raise ValueError("Der Backbone sollte vier Merkmale liefern.")

# Nur wenige Epochen reichen für den Kopf.
loss_history = []
for epoch in range(30):
    model.train()
    optimizer.zero_grad()
    logits = model(train_x)
    loss = loss_function(logits, train_y)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

# Die Testgenauigkeit misst die neue Entscheidung.
model.eval()
with torch.no_grad():
    test_logits = model(test_x)
    predictions = test_logits.argmax(dim=1).numpy()

accuracy = accuracy_score(test_labels, predictions)
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_count = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"Eingefrorene Backbone-Parameter: {frozen_count}")
print(f"Trainierbare Kopf-Parameter: {trainable_count}")
print(f"Testgenauigkeit nach Kopftraining: {accuracy:.2f}")

# Die Kurve zeigt, dass nur der Kopf lernt.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(loss_history) + 1), loss_history, label="Trainingsverlust")
ax.set_title("Klassifikationskopf auf festen Merkmalen")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")
ax.legend()
plt.show()



## **3. Transfer bewerten**

### **3.1. Inferenzzeit messen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_01.jpg?v=1787664394" width="250">



>* Genauigkeit allein reicht nicht zur Modellbewertung
>* Inferenzzeit zeigt praktische Einsatzfähigkeit

>* Messbedingungen kontrollieren und Evaluationsmodus nutzen
>* Mehrfach messen, erste Durchläufe einordnen

>* Inferenzzeit mit Genauigkeit und Kontext abwägen
>* Online- und Offline-Einsatz unterschiedlich bewerten



In [ ]:
#@title Python-Code - Inferenzzeit messen

# Dieses Beispiel misst Inferenzzeit kontrolliert.
# Zwei kleine CNNs werden fair verglichen.
# Die Ausgabe zeigt Geschwindigkeit und Größe.

import time

import numpy as np
import torch
import matplotlib.pyplot as plt

# Wir verwenden nur die CPU für faire Vergleichbarkeit.
torch.set_num_threads(1)

# Ein fester Startwert macht die Messung reproduzierbarer.
torch.manual_seed(42)

# Ein synthetischer Bildstapel ersetzt externe Bilddateien.
batch_size = 16
image_batch = torch.randn(batch_size, 3, 64, 64)

# Beide Modelle lösen dieselbe einfache Klassifikationsaufgabe.
small_model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 8, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
    torch.nn.Linear(8, 2),
)

large_model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 32, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.Conv2d(32, 32, kernel_size=3, padding=1),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d((1, 1)),
    torch.nn.Flatten(),
    torch.nn.Linear(32, 2),
)

# Der Evaluationsmodus ist für Inferenzmessungen wichtig.
small_model.eval()
large_model.eval()

# Diese Funktion misst mehrere Vorwärtsläufe ohne Gradienten.
def measure_model(model, inputs, repeats):
    durations = []
    with torch.inference_mode():
        for _ in range(3):
            _ = model(inputs)
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model(inputs)
            end = time.perf_counter()
            durations.append((end - start) * 1000)
    return np.array(durations)

# Die Parameterzahl ist ein einfacher Größenvergleich.
def count_parameters(model):
    total = 0
    for parameter in model.parameters():
        total += parameter.numel()
    return total

# Mehrere Wiederholungen liefern stabilere typische Werte.
repeats = 30
small_times = measure_model(small_model, image_batch, repeats)
large_times = measure_model(large_model, image_batch, repeats)

# Eine kurze Prüfung verhindert missverständliche Ergebnisse.
if len(small_times) != repeats or len(large_times) != repeats:
    raise ValueError("Die Messreihen haben nicht die erwartete Länge.")

# Wir vergleichen Medianzeit und Modellgröße nebeneinander.
small_median = float(np.median(small_times))
large_median = float(np.median(large_times))
small_params = count_parameters(small_model)
large_params = count_parameters(large_model)

print("Inferenzmessung auf CPU mit synthetischen 64x64-RGB-Bildern.")
print(f"Kleines CNN: {small_median:.3f} ms pro Batch, {small_params} Parameter.")
print(f"Größeres CNN: {large_median:.3f} ms pro Batch, {large_params} Parameter.")
print("Merke: Genauigkeit sollte später gemeinsam mit Laufzeit bewertet werden.")

# Das Balkendiagramm macht den Laufzeitunterschied sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
model_names = ["Kleines CNN", "Größeres CNN"]
median_times = [small_median, large_median]

ax.bar(model_names, median_times, color=["#4C78A8", "#F58518"])
ax.set_title("Gemessene Inferenzzeit pro Batch")
ax.set_xlabel("Modell")
ax.set_ylabel("Medianzeit in Millisekunden")
plt.show()



### **3.2. Genauigkeit vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_02.jpg?v=1787664396" width="250">



>* Modelle nur unter gleichen Bedingungen vergleichen
>* Testdaten strikt für echte Bewertung zurückhalten

>* Gesamtgenauigkeit kann bei Ungleichgewicht täuschen
>* Kritische Klassen und Domänenrobustheit prüfen

>* Genauigkeit durch wiederholte Experimente absichern
>* Modellwahl nach Praxisbalance treffen



In [ ]:
#@title Python-Code - Genauigkeit vergleichen

# Wir vergleichen Genauigkeit unter fairen Bedingungen.
# Ein Testdatensatz bleibt bis zuletzt unberührt.
# Die Grafik zeigt Genauigkeit und Fehlerarten.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import confusion_matrix

# Wir nutzen kleine Bilddaten aus scikit-learn.
digits = load_digits()

# Nur zwei Ziffern machen den Vergleich übersichtlich.
selected_mask = np.isin(digits.target, [3, 8])
X = digits.data[selected_mask]
y = digits.target[selected_mask]

# Diese Prüfung schützt vor unerwarteten Datenformen.
if X.shape[0] < 100 or X.shape[1] != 64:
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Beide Modelle erhalten exakt dieselbe Datenaufteilung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Modell A nutzt alle Pixel als Merkmale.
model_a = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42)
)

# Modell B simuliert einen schwächeren Merkmalsextraktor.
model_b = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42)
)

# Beide Modelle werden nur auf Trainingsdaten angepasst.
model_a.fit(X_train, y_train)
model_b.fit(X_train[:, :32], y_train)

# Der Testdatensatz wird erst jetzt ausgewertet.
pred_a = model_a.predict(X_test)
pred_b = model_b.predict(X_test[:, :32])

# Gesamtgenauigkeit und balancierte Genauigkeit ergänzen sich.
acc_a = accuracy_score(y_test, pred_a)
acc_b = accuracy_score(y_test, pred_b)
bal_a = balanced_accuracy_score(y_test, pred_a)
bal_b = balanced_accuracy_score(y_test, pred_b)

# Die Verwechslungsmatrix zeigt konkrete Fehlerarten.
cm_a = confusion_matrix(y_test, pred_a, labels=[3, 8])
cm_b = confusion_matrix(y_test, pred_b, labels=[3, 8])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testbilder: {len(y_test)} mit denselben Klassen 3 und 8")
print(f"Modell A Genauigkeit: {acc_a:.3f}, balanciert: {bal_a:.3f}")
print(f"Modell B Genauigkeit: {acc_b:.3f}, balanciert: {bal_b:.3f}")
print(f"Fehler A: {len(y_test) - np.trace(cm_a)}, Fehler B: {len(y_test) - np.trace(cm_b)}")

# Die Balken machen den fairen Genauigkeitsvergleich sichtbar.
model_names = ["A: alle Pixel", "B: halbe Pixel"]
accuracies = [acc_a, acc_b]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(model_names, accuracies, color=["#4C78A8", "#F58518"])
ax.set_ylim(0, 1)

ax.set_title("Genauigkeit auf demselben Testdatensatz")
ax.set_xlabel("Verglichenes Modell")
ax.set_ylabel("Testgenauigkeit")

for index, value in enumerate(accuracies):
    ax.text(index, value + 0.02, f"{value:.3f}", ha="center")

plt.show()



### **3.3. Transfer Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_18/Lecture_B/image_03_03.jpg?v=1787664398" width="250">



>* Vortrainierte Modelle realistisch bewerten
>* Genauigkeit gegen Praxistauglichkeit abwägen

>* Modelle fair mit gleichen Daten vergleichen
>* Genauigkeit gegen Geschwindigkeit und Hardware abwägen

>* Offline-Fallback mit lokalen Modellgewichten prüfen
>* Modelle nach Praxis-Kriterien begründet empfehlen



In [ ]:
#@title Python-Code - Transfer Mini Projekt

# Dieses Mini Projekt vergleicht Transfer Varianten.
# Bewertet werden Genauigkeit, Laufzeit und Größe.
# Die Grafik zeigt eine begründete Modellwahl.

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn import __version__ as sklearn_version

# Wir simulieren faire Messwerte für drei Modellvarianten.
model_names = ["Offline Kopf", "Frozen MobileNet", "Feinabstimmung"]
accuracy = np.array([0.82, 0.88, 0.90])
inference_ms = np.array([7.0, 14.0, 31.0])
size_mb = np.array([3.2, 13.5, 45.0])

# Offline Tauglichkeit wird als einfacher Wahrheitswert modelliert.
offline_ready = np.array([True, True, False])
if len(model_names) != len(accuracy):
    raise ValueError("Die Modelllisten müssen gleich lang sein.")

# Ein kleiner Score macht die Abwägung transparent.
accuracy_score = accuracy / accuracy.max()
speed_score = inference_ms.min() / inference_ms
size_score = size_mb.min() / size_mb

# Fehlender Offline Fallback wird deutlich bestraft.
offline_score = offline_ready.astype(float)
combined_score = (
    0.45 * accuracy_score + 0.25 * speed_score
    + 0.15 * size_score + 0.15 * offline_score
)

# Die beste Variante ist die mit dem höchsten Gesamtscore.
best_index = int(np.argmax(combined_score))
best_model = model_names[best_index]

print(f"scikit-learn Version: {sklearn_version}")
print(f"PyTorch Version: {torch.__version__}")
print("Vergleich: Genauigkeit, Inferenzzeit, Größe, Offline")

for index, name in enumerate(model_names):
    offline_text = "ja" if offline_ready[index] else "nein"
    print(f"{name}: {accuracy[index]:.2f}, {inference_ms[index]:.0f} ms, {size_mb[index]:.1f} MB, {offline_text}")

print(f"Empfehlung nach Score: {best_model}")

# Die Balken zeigen den zusammengefassten Praxisscore.
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["tab:blue", "tab:green", "tab:orange"]
ax.bar(model_names, combined_score, color=colors)

# Achsen und Titel machen die Bewertung lesbar.
ax.set_title("Transfer Mini Projekt: Praxisscore")
ax.set_xlabel("Modellvariante")
ax.set_ylabel("Score von 0 bis 1")
ax.set_ylim(0, 1.05)

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Transfer mit PyTorch**</font>


In this lecture, you learned to:
- Setzen Bildaugmentation, Dropout, Batch Normalization, Scheduler und Early Stopping kontrolliert ein. 
- Nutzen ein leichtes vortrainiertes torchvision-Modell als eingefrorenen Merkmalsextraktor. 
- Vergleichen Transfer-Learning-Modelle hinsichtlich Genauigkeit, Laufzeit, Größe und Offline-Fallback. 

In the next Module (Module 19), we will go over 'Sequenzen und Autoencoder'